# Real data preparation, audits, and frozen menus

Formal DiffJev workflow; see AGENTS.md and prompt.md.

In [ ]:
import os, sys, gc
from pathlib import Path
ROOT = Path(os.environ.get('DIFFJEV_ROOT', Path.cwd())).resolve()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
if not (ROOT / 'AGENTS.md').exists() and Path('/content/drive/MyDrive/DiffJev/AGENTS.md').exists():
    ROOT = Path('/content/drive/MyDrive/DiffJev')
assert (ROOT / 'AGENTS.md').exists(), 'Mount Drive and set DIFFJEV_ROOT to the checkout used by notebook 00'
sys.path.insert(0, str(ROOT / 'src'))
from diffjev.notebook_runtime import NotebookRuntime
runtime = NotebookRuntime(ROOT)
runtime.python_code("from diffjev.preparation import check_environment; from pathlib import Path; check_environment(Path.cwd())", label='01_prepare_data_environment')
def run_python(source):
    prefix = '''import gc
from pathlib import Path
from diffjev import workflows as w
from diffjev.config import load_yaml_config
ROOT = w.ROOT
cfg = load_yaml_config(str(ROOT / 'configs/workflow.yaml'))
SEED, DEVICE = cfg['seed'], cfg['device']
'''
    runtime.python_code(prefix + source, label='01_prepare_data')
print('Experiment worker:', runtime.python)


Requires the pinned OpenJev SFT, ToolRet training, original JevBench text cases, ToolRet Queries/Tools snapshots, and the four backbone tokenizer/config snapshots for shared context acceptance (no weights are loaded during preparation). Restore withheld JevBench states from their licensed source with w.restore_benchmark_states(); hashes are checked. Records with ambiguous labels are quarantined. Review data/processed/formal/audit.json before freezing comparisons.

Notebook 00 already includes preparation. This notebook optionally verifies/reuses the same frozen artifacts; it does not overwrite them.

In [ ]:
run_python(r"""
from diffjev.preparation import prepare_all
prepare_all(seed=SEED, scaling_queries=cfg['scaling_queries'])
""")

In [ ]:
run_python(r"""
audit = w.read_json(w.PREPARED/'audit.json')
print('Sources:', audit['sources'])
print('Rejected records:',len(audit['rejections']))
print('Trace:',audit['trace_sha256'])
""")